In [1]:
import gurobipy as gb
import numpy as np
from instance import GRSCInstance

# The GRSC model
* Generalized Reserve Set Covering Problem
* It's based on the RSC, but with the definition of the **minimum quotas of ecological suitability**

In [2]:
instance = GRSCInstance()
grsc = gb.Model("grsc")
grsc.setParam('OutputFlag', 0)

Set parameter Username
Set parameter LicenseID to value 2796100
Academic license - for non-commercial use only - expires 2027-03-23


## The variables

### Land site

* $x_i$, $1 \leq i \leq |V|$ → binary variables defined as:

  $$
  \begin{align*}
  x_i =
  \begin{cases}
  1, & \text{if the land site $i \in V$ is selected}\\
  0, & \text{otherwise}
  \end{cases}
  \end{align*}
  $$


In [3]:
x = grsc.addVars(instance.V, vtype=gb.GRB.BINARY, name="x")

### Core site

* $z_i$, $1 \leq i \leq |V|$ → binary variables defined as:

  $$
  \begin{align*}
  z_i =
  \begin{cases}
  1, & \text{if the land site $i \in V$ is part of the core area}\\
  0, & \text{otherwise}
  \end{cases}
  \end{align*}
  $$

In [4]:
z = grsc.addVars(instance.V, vtype=gb.GRB.BINARY, name="z")

### Specie protection
* $u_s$, $1 \leq s \leq |S|$ → binary variables defined as:

  $$
  \begin{align*}
  u_i =
  \begin{cases}
  1, & \text{if the specie $s \in S$ is protected by the reserve}\\
  0, & \text{otherwise}
  \end{cases}
  \end{align*}
  $$

In [5]:
u = grsc.addVars(instance.S, vtype=gb.GRB.BINARY, name="u")

## Objective function

Function that indicates the cost that needs to be **minimized**.
$$
\gamma(u,x,z) = \sum_{i\in V} c_i x_i
$$

In [6]:
grsc.setObjective(gb.quicksum(instance.c(v) * x[v] for v in instance.V), gb.GRB.MINIMIZE)

## Constraints

### S1-SQ

If a specie in $S_1$ is protected, the habitat suitability score for the land sites part of the core area must be at least the minimum quota of ecological suitability of that specie.
$$
\sum_{i\in V_s} w(i, s)z_i \geq \lambda_s u_s \quad \forall s \in S_1
$$

In [7]:
for s in instance.S_1:
    grsc.addConstr(gb.quicksum(instance.w(v, s) * z[v] for v in instance.v_s(s)) >= instance.l(s) * u[s], name="S1-SQ")


### S2-SQ

Similarly for a specie in $S_2$, considering land sites in the whole reserve.
$$
\sum_{i\in V_s} w(i, s)x_i \geq \lambda_s u_s \quad \forall s \in S_2
$$

In [8]:
for s in instance.S_2:
    grsc.addConstr(gb.quicksum(instance.w(v, s) * x[v] for v in instance.v_s(s)) >= instance.l(s) * u[s], name="S2-SQ")

### S1-PROTECT

At least $P_1$ species of $S_1$ must be protected. 
$$
\sum_{s\in S_1} u_s \geq P_1
$$

In [9]:
grsc.addConstr(gb.quicksum(u[s] for s in instance.S_1) >= instance.P_1, name="S1-PROTECT")

<gurobi.Constr *Awaiting Model Update*>

### S2-PROTECT

At least $P_2$ species of $S_2$ must be protected. 
$$
\sum_{s\in S_2} u_2 \geq P_2
$$

In [10]:
grsc.addConstr(gb.quicksum(u[s] for s in instance.S_2) >= instance.P_2, name="S2-PROTECT")

<gurobi.Constr *Awaiting Model Update*>

### LINK

If a site is in the core, then it must also be in the reserve.
$$
z_i \leq x_i \quad \forall i \in V
$$

In [11]:
for v in instance.V:
    grsc.addConstr(z[v] <= x[v], name=f"LINK")   

In [ ]:

grsc.optimize()

# LA STAMPA è DA SISTEMARE PERCHé è BRUTTA, magari aggiungendo il grafico, MA PER ORA VA BENE COSì

print("Status:", grsc.Status)
print("Objective:", grsc.ObjVal)

print("\n--- Variabili x ---")
for v in instance.V:
    print(f"  x[{v}] = {x[v].X}")

print("\n--- Variabili z ---")
for v in instance.V:
    print(f"  z[{v}] = {z[v].X}")

print("\n--- Variabili u ---")
for s in instance.S:
    print(f"  u[{s}] = {u[s].X}")

Status: 2
Objective: 3.0

--- Variabili x ---
  x[0] = 1.0
  x[1] = 1.0
  x[2] = 1.0
  x[3] = 0.0
  x[4] = 0.0

--- Variabili z ---
  z[0] = 1.0
  z[1] = 1.0
  z[2] = 1.0
  z[3] = 0.0
  z[4] = 0.0

--- Variabili u ---
  u[0] = 1.0
  u[1] = 1.0
  u[2] = 1.0
  u[3] = 1.0
  u[4] = 1.0
  u[5] = 1.0
  u[6] = 1.0
  u[7] = 1.0
  u[8] = 1.0
  u[9] = 1.0
  u[10] = 1.0
  u[11] = 1.0
